In [ ]:
import pandas as pd
df = pd.read_csv("loan_encoded.csv")
df.head(10)

In [ ]:
df.grade.value_counts()

In [ ]:
# 基线模型：分层切分 + 5折交叉验证 + 全训练集重训后测试（低内存安全版）
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from joblib import parallel_backend
import numpy as np
import matplotlib.pyplot as plt

# 4核16GB推荐：外层CV串行，内部线程并行
CV_N_JOBS = 1
RF_N_JOBS = 2

def run_rf_experiment(X_train_full, y_train_full, X_test, y_test, model_params, cv, model_label):
    # 用 threading backend 降低多进程复制数据造成的内存峰值
    with parallel_backend('threading'):
        cv_scores = cross_validate(
            RandomForestClassifier(**model_params),
            X_train_full, y_train_full,
            cv=cv,
            scoring=['accuracy', 'f1_weighted'],
            n_jobs=CV_N_JOBS,
            pre_dispatch='1*n_jobs'
        )
    print(f"{model_label}CV准确率: {cv_scores['test_accuracy'].mean():.4f} +/- {cv_scores['test_accuracy'].std():.4f}")
    print(f"{model_label}CV加权F1: {cv_scores['test_f1_weighted'].mean():.4f} +/- {cv_scores['test_f1_weighted'].std():.4f}")

    model = RandomForestClassifier(**model_params)
    model.fit(X_train_full, y_train_full)

    y_test_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    print(f"{model_label}测试集准确率: {test_accuracy:.4f}")
    print(f"{model_label}测试集加权F1: {test_f1:.4f}")

    return model, cv_scores, y_test_pred, test_accuracy, test_f1

def plot_top_feature_importances(model, feature_columns, title, color='b', top_n=20):
    feature_importances = model.feature_importances_
    indices = np.argsort(feature_importances)[::-1][:top_n]

    plt.figure(figsize=(8, 6))
    plt.title(title)
    plt.barh(range(len(indices)), feature_importances[indices], color=color, align='center')
    plt.yticks(range(len(indices)), [feature_columns[i] for i in indices])
    plt.xlabel('Relative Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

X = df.drop('grade', axis=1)
y = df['grade']

# 切分测试集（10%），训练池用于交叉验证和最终重训
X_train_full, X_test_base, y_train_full, y_test_base = train_test_split(
    X, y,
    test_size=0.1,
    random_state=123,
    stratify=y
)

model_params = dict(
    n_estimators=132,
    criterion='entropy',
    max_depth=39,
    max_features='sqrt',
    min_samples_leaf=2,
    min_samples_split=6,
    random_state=111,
    class_weight='balanced',
    n_jobs=RF_N_JOBS
 )

# 5折分层交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

rf_base, cv_scores_base, y_test_pred_base, test_accuracy_base, test_f1_base = run_rf_experiment(
    X_train_full, y_train_full, X_test_base, y_test_base, model_params, cv, model_label='基线模型'
 )

plot_top_feature_importances(
    rf_base,
    X_train_full.columns,
    title='Top Feature Importances (Baseline)',
    color='b',
    top_n=20
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 基线模型测试集混淆矩阵
cm_base = confusion_matrix(y_test_base, y_test_pred_base)
disp_base = ConfusionMatrixDisplay(confusion_matrix=cm_base)
disp_base.plot(cmap='Blues', values_format='d')
plt.title('Baseline Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print('基线模型分类报告:')
print(classification_report(y_test_base, y_test_pred_base, digits=4))

In [ ]:
# 自动分批删减 + 早停规则（先20万分层样本，再全量复核）
from copy import deepcopy

# 早停参数（可按需求调整）
BATCH_SIZE = 2                 # 每轮新增删除特征数
MAX_REMOVE = 20                # 最多删除特征数
F1_DROP_TOL = 0.002            # 允许的test_f1下降阈值
USE_CV_GATE = True             # 同时要求cv_f1也不过度下降
CV_DROP_TOL = 0.002

# 采样策略：先在20万分层样本上做ablation
USE_ABLATION_SAMPLE = True
ABLATION_SAMPLE_SIZE = 200_000

# 如果有业务上必须保留的特征，可写在这里
MUST_KEEP = set([])

if USE_ABLATION_SAMPLE and len(df) > ABLATION_SAMPLE_SIZE:
    _, df_ablation = train_test_split(
        df,
        test_size=ABLATION_SAMPLE_SIZE,
        random_state=123,
        stratify=df['grade']
    )
    print(f'使用分层样本做ablation: {len(df_ablation)} 行')
else:
    df_ablation = df.copy()
    print(f'使用全量数据做ablation: {len(df_ablation)} 行')

# 候选删除顺序：按基线模型Gini重要性从低到高
importance_rank = pd.Series(rf_base.feature_importances_, index=X_train_full.columns).sort_values()
candidate_cols = [c for c in importance_rank.index if c not in MUST_KEEP]

# 基线参考分数（来自第5个单元）
base_test_f1 = test_f1_base
base_cv_f1 = cv_scores_base['test_f1_weighted'].mean()

records = []
best_drop_cols = []
best_test_f1 = -1.0
best_cv_f1 = -1.0

for k in range(BATCH_SIZE, min(MAX_REMOVE, len(candidate_cols)) + 1, BATCH_SIZE):
    drop_cols = candidate_cols[:k]
    Xk = df_ablation.drop(columns=drop_cols + ['grade'])
    yk = df_ablation['grade']

    Xk_train_full, Xk_test, yk_train_full, yk_test = train_test_split(
        Xk, yk,
        test_size=0.1,
        random_state=123,
        stratify=yk
    )

    _, cv_scores_k, _, test_acc_k, test_f1_k = run_rf_experiment(
        Xk_train_full, yk_train_full, Xk_test, yk_test,
        deepcopy(model_params), cv,
        model_label=f'样本ablation-删减{k}个特征模型'
    )
    cv_f1_k = cv_scores_k['test_f1_weighted'].mean()

    records.append({
        'n_removed': k,
        'removed_features': drop_cols,
        'cv_f1_mean': cv_f1_k,
        'test_accuracy': test_acc_k,
        'test_f1_weighted': test_f1_k
    })

    # 更新当前最优（先看test_f1，再看cv_f1）
    if (test_f1_k > best_test_f1) or (np.isclose(test_f1_k, best_test_f1) and cv_f1_k > best_cv_f1):
        best_test_f1 = test_f1_k
        best_cv_f1 = cv_f1_k
        best_drop_cols = drop_cols

    # 早停：相对基线明显下降则停止（仅在用全量ablation时严格比较基线）
    if len(df_ablation) == len(df):
        test_drop_too_much = (base_test_f1 - test_f1_k) > F1_DROP_TOL
        cv_drop_too_much = (base_cv_f1 - cv_f1_k) > CV_DROP_TOL
        if test_drop_too_much and ((not USE_CV_GATE) or cv_drop_too_much):
            print(f'触发早停：删除{k}个特征后性能下降超过阈值。')
            break

ablation_df = pd.DataFrame(records)
display_cols = ['n_removed', 'cv_f1_mean', 'test_accuracy', 'test_f1_weighted']
if not ablation_df.empty:
    print('样本ablation结果（按test_f1从高到低）：')
    display(ablation_df[display_cols].sort_values('test_f1_weighted', ascending=False).reset_index(drop=True))
    print('样本阶段建议删除的特征:')
    print(best_drop_cols)
else:
    print('未生成ablation结果，请检查参数设置。')

# 全量复核：只对样本阶段最优方案做一次全量评估
selected_drop_cols = best_drop_cols
if len(selected_drop_cols) > 0 and len(df_ablation) < len(df):
    print('开始全量复核样本阶段最优删减方案...')
    Xv = df.drop(columns=selected_drop_cols + ['grade'])
    yv = df['grade']
    Xv_train_full, Xv_test, yv_train_full, yv_test = train_test_split(
        Xv, yv,
        test_size=0.1,
        random_state=123,
        stratify=yv
    )
    _, cv_scores_v, _, test_acc_v, test_f1_v = run_rf_experiment(
        Xv_train_full, yv_train_full, Xv_test, yv_test,
        deepcopy(model_params), cv,
        model_label='全量复核-样本最优删减模型'
    )
    cv_f1_v = cv_scores_v['test_f1_weighted'].mean()
    print(f'全量复核 test_f1={test_f1_v:.4f}, 基线 test_f1={base_test_f1:.4f}')
    print(f'全量复核 cv_f1={cv_f1_v:.4f}, 基线 cv_f1={base_cv_f1:.4f}')

In [ ]:
# 学习曲线 + 参数范围收窄（快速诊断版）
from sklearn.model_selection import learning_curve, validation_curve

# 诊断采样（分层20万）
DIAG_SAMPLE_SIZE = 200_000
if len(df) > DIAG_SAMPLE_SIZE:
    _, df_diag = train_test_split(
        df,
        test_size=DIAG_SAMPLE_SIZE,
        random_state=123,
        stratify=df['grade']
    )
else:
    df_diag = df.copy()

X_diag = df_diag.drop('grade', axis=1)
y_diag = df_diag['grade']

cv_quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=123)
rf_diag = RandomForestClassifier(**model_params)

# 1) 学习曲线：看是否已经接近平台
train_sizes, train_scores, valid_scores = learning_curve(
    estimator=rf_diag,
    X=X_diag,
    y=y_diag,
    train_sizes=[0.1, 0.3, 0.5, 0.7, 1.0],
    cv=cv_quick,
    scoring='f1_weighted',
    n_jobs=1
 )

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)

plt.figure(figsize=(7, 4))
plt.plot(train_sizes, train_mean, marker='o', label='train_f1')
plt.plot(train_sizes, valid_mean, marker='o', label='valid_f1')
plt.title('Learning Curve (Weighted F1)')
plt.xlabel('Training Samples')
plt.ylabel('Weighted F1')
plt.legend()
plt.tight_layout()
plt.show()

if len(valid_mean) >= 2:
    gain_tail = valid_mean[-1] - valid_mean[-2]
    print(f'学习曲线末段提升: {gain_tail:.5f}')
    if gain_tail < 0.001:
        print('结论: 学习曲线接近平台，可优先缩小参数搜索范围。')
    else:
        print('结论: 仍有提升空间，可保留适度搜索。')

# 2) n_estimators 验证曲线：定位树数平台
n_grid = [40, 80, 120, 160, 220]
tr_n, va_n = validation_curve(
    estimator=RandomForestClassifier(
        criterion=model_params['criterion'],
        max_depth=model_params['max_depth'],
        max_features=model_params['max_features'],
        min_samples_leaf=model_params['min_samples_leaf'],
        min_samples_split=model_params['min_samples_split'],
        random_state=model_params['random_state'],
        class_weight=model_params['class_weight'],
        n_jobs=RF_N_JOBS
    ),
    X=X_diag,
    y=y_diag,
    param_name='n_estimators',
    param_range=n_grid,
    cv=cv_quick,
    scoring='f1_weighted',
    n_jobs=1
 )

n_valid_mean = va_n.mean(axis=1)
plt.figure(figsize=(7, 4))
plt.plot(n_grid, n_valid_mean, marker='o')
plt.title('Validation Curve: n_estimators')
plt.xlabel('n_estimators')
plt.ylabel('CV Weighted F1')
plt.tight_layout()
plt.show()

best_idx = int(np.argmax(n_valid_mean))
best_n = n_grid[best_idx]
print(f'n_estimators候选最优: {best_n}')

# 根据曲线给出收窄范围（可直接用于下一轮RandomizedSearch）
n_low = max(40, best_n - 40)
n_high = min(260, best_n + 40)
suggested_param_space = {
    'n_estimators': (n_low, n_high),
    'max_depth': (16, 40),
    'min_samples_split': (4, 20),
    'min_samples_leaf': (2, 12),
    'max_features': ['sqrt', 0.3, 0.5],
    'criterion': ['gini', 'entropy']
}
print('建议下一轮搜索空间:')
print(suggested_param_space)

In [ ]:
# 改进模型：使用自动删减结果；若不存在则回退到手动删两列
default_drop_cols = ['addr_state_', 'purpose_']
drop_cols_final = selected_drop_cols if ('selected_drop_cols' in globals() and len(selected_drop_cols) > 0) else default_drop_cols
print(f'本轮删除特征数: {len(drop_cols_final)}')
print('删除特征列表:', drop_cols_final)

df2 = df.drop(drop_cols_final, axis=1)

X2 = df2.drop('grade', axis=1)
y2 = df2['grade']

# 同样采用分层切分，保证与基线可比
X2_train_full, X2_test, y2_train_full, y2_test = train_test_split(
    X2, y2,
    test_size=0.1,
    random_state=123,
    stratify=y2
)

rf_drop, cv_scores_drop, y2_test_pred, test_accuracy_drop, test_f1_drop = run_rf_experiment(
    X2_train_full, y2_train_full, X2_test, y2_test, model_params, cv, model_label='改进模型'
 )

plot_top_feature_importances(
    rf_drop,
    X2_train_full.columns,
    title='Top Feature Importances (Drop Features by Ablation)',
    color='g',
    top_n=20
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_drop = confusion_matrix(y2_test, y2_test_pred)
disp_drop = ConfusionMatrixDisplay(confusion_matrix=cm_drop)
disp_drop.plot(cmap='Greens', values_format='d')
plt.title('Drop-Features Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print('改进模型分类报告:')
print(classification_report(y2_test, y2_test_pred, digits=4))

comparison = pd.DataFrame({
    'model': ['baseline', 'drop_addr_state_purpose'],
    'test_accuracy': [test_accuracy_base, test_accuracy_drop],
    'test_f1_weighted': [test_f1_base, test_f1_drop],
    'cv_accuracy_mean': [cv_scores_base['test_accuracy'].mean(), cv_scores_drop['test_accuracy'].mean()],
    'cv_f1_mean': [cv_scores_base['test_f1_weighted'].mean(), cv_scores_drop['test_f1_weighted'].mean()]
})

comparison.sort_values(by='test_f1_weighted', ascending=False)